## Importing Libraries 

In [1]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"

### KNOWLEDGE BASE 
* In a real RAG system this would be chunks from your documents.
* Here it's a hardcoded list of backend engineering concepts.
* Each string is one "document" — a self-contained piece of knowledge.

In [2]:
KNOWLEDGE_BASE = [
    "PostgreSQL B-tree indexes support equality and range queries. Create with CREATE INDEX idx_name ON table(column). Use CONCURRENTLY to avoid table locks in production.",
    "Composite indexes in PostgreSQL cover multiple columns. Column order matters — leftmost column must appear in WHERE clause for the index to be used.",
    "Redis cache-aside pattern: check cache first, on miss query DB, store result in cache with TTL. Invalidate cache on writes to prevent stale data.",
    "JWT access tokens should be short-lived (15-60 minutes). Refresh tokens rotate on each use. Store refresh tokens in DB to enable revocation.",
    "FastAPI dependency injection uses Depends(). Dependencies can be chained — a dependency can itself depend on other dependencies. Async dependencies use async def.",
    "PostgreSQL EXPLAIN ANALYZE shows query execution plan. Seq Scan = full table scan. Index Scan = uses index. Always run before and after adding indexes.",
    "Docker multi-stage builds reduce image size by separating build and runtime stages. Only copy artifacts needed to run, not build tools.",
    "Kubernetes pods are the smallest deployable unit. Each pod gets its own IP. Services provide stable networking on top of pods which can die and restart.",
    "Database connection pooling maintains a pool of reuse connections instead of opening/closing per request. SQLAlchemy pool_size=5, max_overflow=10 are common defaults.",
    "Rate limiting with Redis sliding window: store request timestamps in a sorted set per client IP, remove old timestamps, count remaining. O(log n) per request.",
    "Multi-tenant SaaS: every query must filter by tenant_id. Use row-level security or application-level filtering. Never trust client-supplied tenant_id without auth check.",
    "ACID properties: Atomicity (all or nothing), Consistency (valid state), Isolation (transactions don't interfere), Durability (committed data survives crashes).",
    "Async Python with FastAPI: use async def for I/O-bound operations (DB queries, HTTP calls, Redis). Use sync def for CPU-bound work. Mixing incorrectly blocks the event loop.",
    "GitHub Actions CI pipeline: trigger on push/PR, run linting (flake8/black), tests (pytest), build Docker image, push to registry. Fail fast — lint before tests.",
    "PostgreSQL full-text search: tsvector stores processed document, tsquery matches against it. Use GIN index on tsvector column for fast search. to_tsvector() + to_tsquery().",
]

### EMBEDDING FUNCTIONS

In [3]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    """
    Embed a list of texts in a single API call.
    Always batch — one API call for N texts, not N API calls.
    Returns list of vectors in same order as input.
    """
    response = client.embeddings.create(
        input=texts, 
        model=EMBEDDING_MODEL
        )
    return [data.embedding for data in response.data]

def embed_query(query: str) -> list[float]:
    """
    Embed a single query string.
    Separate function because queries are always single strings.
    """
    response = client.embeddings.create(
        input=query, 
        model=EMBEDDING_MODEL
        )
    return response.data[0].embedding

### SIMILARITY

In [4]:
def cosine_similarity(vecA: list[float], vecB: list[float]) -> float:
    """
    Cosine similarity between two vectors.
    Result: 1.0 = identical, 0.0 = unrelated, -1.0 = opposite.
    We use numpy for efficient dot product and norm computation.
    """
    a_srray = np.array(vecA)
    b_array = np.array(vecB)
    dot_product = np.dot(a_srray, b_array)
    norm = np.linalg.norm(a_srray) * np.linalg.norm(b_array)
    if norm == 0:
        return 0.0
    return float( dot_product / norm )

### SEARCH

In [5]:
def search(
        query: str,
        doc_embeddings: list[list[float]],
        top_k: int = 3,
) -> list[dict]:
    """
    Find the top_k most semantically similar documents to the query.

    Process:
    1. Embed the query (one API call)
    2. Compute cosine similarity against every pre-computed doc embedding
    3. Sort by similarity descending
    4. Return top_k results with score and text

    Note: doc_embeddings are pre-computed at startup — no per-query DB hit.
    This is exactly what a vector database does at scale.
    """
    query_embedding = embed_query(query)
    scored_docs = []
    for i, doc_embedding in enumerate(doc_embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        scored_docs.append({
            "score": score, 
            "text": KNOWLEDGE_BASE[i],
            "index": i,
            })
    scored_docs.sort(key=lambda x: x["score"], reverse=True)
    return scored_docs[:top_k]

### MAIN
* Pre-compute all document embeddings at startup, one batch API call.
* In production this would be loaded from a vector DB, not recomputed.

*Demonstrate similarity scores directly*

*Interactive search loop*

*Interactive mode*

In [6]:
def main():
    print("="*65)
    print("BACKEND ENGINEERING SEMANTIC SEARCH")
    print(f"Model: {EMBEDDING_MODEL} | Knowledge Base Size: {len(KNOWLEDGE_BASE)} documents")
    print("="*65)

    print("\nEmbedding knowledge base...", end="", flush=True)
    doc_embeddings = embed_texts(KNOWLEDGE_BASE)
    print(f"Done. {len(doc_embeddings)} vectors created, {len(doc_embeddings[0])} dimensions each. \n")

    print("── Direct similarity comparison ────────────────────────────")

    pairs = [
        ("PostgreSQL index optimization", "B-tree indexes in PostgreSQL for fast queries"),
        ("PostgreSQL index optimization", "How to bake sourdough bread"),
        ("JWT authentication security", "access tokens and refresh tokens"),
    ]

    for textA, textB in pairs:
        vecs = embed_texts([textA, textB])
        score = cosine_similarity(vecs[0], vecs[1])
        print(f" '{textA[:35]}...' vs")
        print(f" '{textB[:35]}...'") 
        print(f" => Similarity Score: {score:.4f}\n")

    print("── Semantic search results ───────────────────────────────")
    test_queries = [
        "How do I make my database queries faster?",
        "authentication and token security",
        "running containers in production",
        "caching to reduce database load",
    ]

    print("Running demo queries first...\n")

    for query in test_queries:
        print(f"Query: \"{query}\"")
        results = search(query, doc_embeddings, top_k=3)
        for i, result in enumerate(results, 1):
            preview  = result['text'][:90] + "..." if len(result['text']) > 90 else result['text']
            print(f" {i}. (Score: {result['score']:.4f}) {preview}")
        print("\n")


    print("\nNow you can enter your own queries!")
    while True:
        try:
            query = input("Enter a query (or 'q'/'quit'/'exit' to quit): ")
        except (KeyboardInterrupt, EOFError):
            print("\nExiting. Goodbye!")
            break
        if not query or query.lower() in ['q', 'quit', 'exit']:
            print("\nExiting. Goodbye!")
            break

        results = search(query, doc_embeddings, top_k=3)

        print(f"\nTop {len(results)} results for: \"{query}\"")
        for i, result in enumerate(results, 1):
            preview  = result['text'][:100] + "..." if len(result['text']) > 100 else result['text']
            print(f" {i}. (Score: {result['score']:.4f}) {preview}")

    print("All done!")
    

# Run

In [8]:
if __name__ == "__main__":
    main()

BACKEND ENGINEERING SEMANTIC SEARCH
Model: text-embedding-3-small | Knowledge Base Size: 15 documents

Embedding knowledge base...Done. 15 vectors created, 1536 dimensions each. 

── Direct similarity comparison ────────────────────────────
 'PostgreSQL index optimization...' vs
 'B-tree indexes in PostgreSQL for fa...'
 => Similarity Score: 0.7046

 'PostgreSQL index optimization...' vs
 'How to bake sourdough bread...'
 => Similarity Score: 0.0634

 'JWT authentication security...' vs
 'access tokens and refresh tokens...'
 => Similarity Score: 0.4429

── Semantic search results ───────────────────────────────
Running demo queries first...

Query: "How do I make my database queries faster?"
 1. (Score: 0.3897) Redis cache-aside pattern: check cache first, on miss query DB, store result in cache with...
 2. (Score: 0.3869) PostgreSQL B-tree indexes support equality and range queries. Create with CREATE INDEX idx...
 3. (Score: 0.3710) PostgreSQL EXPLAIN ANALYZE shows query execution p